# Install Package

In [ ]:
# 1. Nang cap cong cu loi
!pip install -U pip "setuptools<82.0.0" "jedi>=0.16"

# 2. Cai dat he sinh thai Unsloth & Xformers
!pip install unsloth==2026.5.2 unsloth_zoo xformers

# 3. Cai dat TRL (SFT Trainer)
!pip install git+https://github.com/huggingface/trl.git@main --no-deps

# Import Library

In [ ]:
import os
import re
import torch
from trl import SFTConfig, SFTTrainer
from unsloth import FastLanguageModel

# Load BaseModel

In [3]:
# Bật tính năng Standby của Unsloth để tiết kiệm 30%+ bộ nhớ cho RL
os.environ['UNSLOTH_VLLM_STANDBY'] = "1"

max_seq_length = 2048 # Có thể tăng lên nếu suy luận dài
lora_rank = 32 # Rank cao hơn = thông minh hơn, nhưng train chậm hơn

# Tải mô hình cơ sở
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "deepseek-ai/DeepSeek-R1-0528-Qwen3-8B", # Tên model LLM
    max_seq_length = max_seq_length,  # độ dài của context length mà mô hình có thể đọc và sinh ra trong 1 chu kỳ
    load_in_4bit = True,               # quantization xuống 4-bit
    fast_inference = False,           # tắt suy luận nhanh để ổn định hệ thống
    max_lora_rank = lora_rank,        # rank của LoRA
    load_in_fp8 = False,               # tắt chế độ tải mô hình bằng định dạng độ chính xác Float8
)
# Cấu hình LoRA Adapter
model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ], # chọn module của model để áp dụng LoRA
    lora_alpha = lora_rank * 2,
    use_gradient_checkpointing = "unsloth",  # dọn dẹp bộ nhớ rác sinh ra khi train giảm VRAM
    random_state = 3407,
)

Unrecognized keys in `rope_parameters` for 'rope_type'='yarn': {'attn_factor'}


==((====))==  Unsloth 2026.5.2: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unrecognized keys in `rope_parameters` for 'rope_type'='yarn': {'attn_factor'}
Unrecognized keys in `rope_parameters` for 'rope_type'='yarn': {'attn_factor'}


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

special_tokens_map.json:   0%|          | 0.00/472 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Unrecognized keys in `rope_parameters` for 'rope_type'='yarn': {'attn_factor'}


unsloth/deepseek-r1-0528-qwen3-8b-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.5.2 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


# Chat template

In [ ]:
# System prompt cho EXACT 2026
system_prompt = (
    "You are an expert educational AI assistant for the EXACT 2026 competition. "
    "For logic problems: analyze premises carefully, apply formal reasoning, "
    "and derive the correct conclusion. "
    "For physics problems: identify relevant formulas, show step-by-step calculations, "
    "and provide the final numerical answer with correct units. "
    "Always think step-by-step inside <think>...</think> tags, "
    "then give your final answer inside <answer>...</answer> tags."
)

# Khong can sua chat_template vi SFT Trainer tu dong xu ly conversations format

# Dataset

In [ ]:
from datasets import load_dataset

# ====================================================================
# HUONG DAN: Upload file train.jsonl va val.jsonl len Google Drive
# vao thu muc /content/drive/MyDrive/exact2026_data/
# ====================================================================

# Mount Google Drive (neu chua mount)
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

# Load dataset EXACT 2026 tu Google Drive
DRIVE_DATA_DIR = "/content/drive/MyDrive/exact2026_data"

train_dataset = load_dataset(
    "json",
    data_files=f"{DRIVE_DATA_DIR}/train.jsonl",
    split="train"
)

# Val dataset (optional - de danh gia)
try:
    val_dataset = load_dataset(
        "json",
        data_files=f"{DRIVE_DATA_DIR}/val.jsonl",
        split="train"
    )
    print(f"Val dataset: {len(val_dataset)} samples")
except:
    val_dataset = None
    print("No val dataset found, skipping.")

print(f"Train dataset: {len(train_dataset)} samples")
print(f"Sample: {train_dataset[0]['conversations'][:2]}")

# REWARD FUNCTIONS



In [ ]:
# [SFT] Khong can reward function nhu GRPO
# Reward function da bi bo khi chuyen tu GRPO sang SFT
# SFT hoc truc tiep tu cap (input, output) trong dataset

# Config

In [7]:
# !pip install vllm

In [ ]:
# ====================================================================
# CAU HINH SFT TRAINER
# ====================================================================
# Co che chong mat du lieu khi Colab disconnect:
#   - save_steps = 50: Tu dong luu checkpoint moi 50 buoc
#   - save_total_limit = 3: Chi giu 3 checkpoint moi nhat
#   - Khi bi disconnect: copy thu muc checkpoint sang nick khac
#     roi tiep tuc train bang resume_from_checkpoint
# ====================================================================

training_args = SFTConfig(
    output_dir = "/content/drive/MyDrive/sft_checkpoints",  # Luu checkpoint vao Google Drive
    save_steps = 50,                                        # Tu dong luu moi 50 buoc
    save_total_limit = 3,                                   # Chi giu 3 checkpoint moi nhat

    # Hyperparameters
    learning_rate = 2e-5,                                   # SFT dung lr cao hon GRPO
    weight_decay = 0.01,
    bf16 = False,
    fp16 = False,
    max_grad_norm = 1.0,
    warmup_steps = 10,
    lr_scheduler_type = "cosine",                           # cosine tot hon linear cho SFT
    optim = "adamw_8bit",                                   # 8bit optimizer giam VRAM

    # Batch size
    logging_steps = 1,
    per_device_train_batch_size = 2,                        # Tang neu du VRAM
    gradient_accumulation_steps = 4,                        # Effective batch = 8

    # Training length
    num_train_epochs = 3,                                   # 3 epoch qua 2160 samples
    max_seq_length = 2048,                                  # Max sequence length

    # Dataset format
    dataset_text_field = None,                              # None vi dung conversations format
    packing = False,                                        # Tat packing de giu nguyen cau truc conversation

    report_to = "none",
    seed = 3407,
)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    args = training_args,
    train_dataset = train_dataset,
    eval_dataset = val_dataset,                             # Optional: danh gia tren val set
)

In [28]:
import os
os.environ["ACCELERATE_MIXED_PRECISION"] = "no"

In [ ]:
# ====================================================================
# CHAY TRAINING
# Neu tiep tuc tu checkpoint cu (sau khi Colab disconnect):
#   trainer.train(resume_from_checkpoint=True)
# Neu train tu dau:
#   trainer.train()
# ====================================================================

trainer.train()

In [ ]:
from unsloth import FastLanguageModel

# Giả sử 'model' và 'tokenizer' là cái ông đã load thành công ở bước [3]
# Thử convert sang GGUF bản 4-bit (phổ biến nhất)
model.save_pretrained_gguf(
    "test_model_gguf",
    tokenizer,
    quantization_method = "q4_k_m"
)